# Weather Data Q&A Pipeline
This notebook walks through the end-to-end Data Q&A process for the `train.csv` weather dataset. We move from simple descriptive statistics all the way to predictive machine learning and prescriptive rule extraction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv('train.csv')
df.head()

### Phase 1: Descriptive Q&A (The Baseline)
**Question:** What is our baseline reality for this dataset?

In [ ]:
print(f"Total recorded days: {len(df)}")
print("\nRainfall Distribution (Baseline):")
print(df['rainfall'].value_counts(normalize=True) * 100)

print("\nBasic Statistics for Weather Metrics:")
df.describe()

### Phase 2: Diagnostic Q&A (The 'Why')
**Question:** What are the undeniable drivers of rain in this location?

In [ ]:
# Calculate correlation with rainfall
corr = df.corr()['rainfall'].sort_values(ascending=False)
print("Correlation with Rainfall:")
print(corr)

# Plotting correlation
plt.figure(figsize=(10, 6))
sns.barplot(x=corr.values, y=corr.index, palette='viridis')
plt.title('Feature Correlation with Rainfall')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

### Phase 3: Exploratory Q&A (Hidden Patterns)
**Question:** Is there a 'rainy season' or a 'dry season'?

In [ ]:
# Create pseudo-months (assuming ~30.5 days per month for 365 days)
df['month'] = ((df['day'] - 1) // 30.5) + 1
df['month'] = df['month'].astype(int)

monthly_rain = df.groupby('month')['rainfall'].mean() * 100

# Plot the seasonality
plt.figure(figsize=(10, 5))
sns.barplot(x=monthly_rain.index, y=monthly_rain.values, palette='Blues_d')
plt.title('Probability of Rain by Month')
plt.ylabel('Rainfall Probability (%)')
plt.xlabel('Month')
plt.tight_layout()
plt.show()

### Phase 4: Predictive Q&A (Forecasting)
**Question:** If we know today's metrics, how accurately can we predict tomorrow's rain using an AI model?

In [ ]:
# Define features (X) and target (y)
X = df.drop(columns=['id', 'day', 'month', 'rainfall'])
y = df['rainfall']

# Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Make predictions on the unseen test set
y_pred = rf.predict(X_test)

# Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"Random Forest Accuracy: {acc * 100:.2f}%")

# View Feature Importances
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importances:")
print(importances.round(4))

### Phase 5: Prescriptive Q&A (The Golden Rules)
**Question:** Can we boil this complex AI model down into simple, human-readable rules?

In [ ]:
# Train a shallow Decision Tree to extract readable thresholds
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

# Export the rules
tree_rules = export_text(dt, feature_names=list(X.columns))
print("Decision Tree Logic (Max Depth 3):\n")
print(tree_rules)